In [1]:
# Importaciones
import kagglehub
import pandas as pd
import openpyxl
import os
import re

# Descargar el dataset de Kaggle
path = kagglehub.dataset_download("ranadeep/credit-risk-dataset")

print("Path to dataset files:", path)
print(os.listdir(path))

c:\Users\srand\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\srand\.cache\kagglehub\datasets\ranadeep\credit-risk-dataset\versions\3
['LCDataDictionary.xlsx', 'loan']


In [2]:
# Cargar diccionario de datos y dataset de creditos
data_dictionary = pd.read_excel(path + "/LCDataDictionary.xlsx")
data_loan = pd.read_csv(path + "/loan/loan.csv", low_memory=False)

print("Diccionario de datos:")
print(data_dictionary.head())
print("\nColumnas del dataset:")
print(data_loan.columns.tolist())
print("\nInformacion general:")
print(data_loan.info())

c:\Users\srand\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Diccionario de datos:
               LoanStatNew                                        Description
0               addr_state  The state provided by the borrower in the loan...
1               annual_inc  The self-reported annual income provided by th...
2         annual_inc_joint  The combined self-reported annual income provi...
3         application_type  Indicates whether the loan is an individual ap...
4  collection_recovery_fee                     post charge off collection fee

Columnas del dataset:
['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'init

In [3]:
# Ver cuantos nulos tiene cada columna
print("Nulos por columna:")
print(data_loan.isnull().sum())

# Ver valores unicos de la variable objetivo
print("\nValores unicos en loan_status:")
print(data_loan["loan_status"].value_counts())

Nulos por columna:
id                       0
member_id                0
loan_amnt                0
funded_amnt              0
funded_amnt_inv          0
                     ...  
all_util            866007
total_rev_hi_lim     70276
inq_fi              866007
total_cu_tl         866007
inq_last_12m        866007
Length: 74, dtype: int64

Valores unicos en loan_status:
loan_status
Current                                                601779
Fully Paid                                             207723
Charged Off                                             45248
Late (31-120 days)                                      11591
Issued                                                   8460
In Grace Period                                          6253
Late (16-30 days)                                        2357
Does not meet the credit policy. Status:Fully Paid       1988
Default                                                  1219
Does not meet the credit policy. Status:Charged Off      

In [4]:
# Seleccion de columnas relevantes para el modelo
# Se excluyen columnas administrativas, de texto libre y con informacion posterior al credito
columnas_seleccionadas = [
    "loan_amnt", "funded_amnt", "funded_amnt_inv", "term", "int_rate",
    "installment", "emp_title", "emp_length", "home_ownership", "annual_inc",
    "verification_status", "purpose", "zip_code", "addr_state", "dti",
    "delinq_2yrs", "inq_last_6mths", "mths_since_last_delinq",
    "mths_since_last_record", "open_acc", "pub_rec", "revol_bal", "revol_util",
    "total_acc", "initial_list_status", "collections_12_mths_ex_med",
    "mths_since_last_major_derog", "application_type", "annual_inc_joint",
    "dti_joint", "verification_status_joint", "acc_now_delinq", "tot_coll_amt",
    "tot_cur_bal", "open_acc_6m", "open_il_6m", "open_il_12m", "open_il_24m",
    "mths_since_rcnt_il", "total_bal_il", "il_util", "open_rv_12m", "open_rv_24m",
    "max_bal_bc", "all_util", "total_rev_hi_lim", "inq_fi", "total_cu_tl",
    "inq_last_12m",
]

In [5]:
# Calcular porcentaje de nulos por columna
nulos = data_loan[columnas_seleccionadas].isnull().sum()
pct_nulos = (nulos / len(data_loan) * 100).sort_values(ascending=False)

print("Columnas con mas del 30% de nulos:")
print(pct_nulos[pct_nulos > 30])

# Eliminar columnas con mas del 85% de nulos
# Nota: mths_since_last_delinq, mths_since_last_record, mths_since_last_major_derog
# y mths_since_rcnt_il NO se eliminan aunque tengan muchos nulos, ya que el hecho
# de que sean nulas indica que el evento nunca ocurrio, lo cual es informacion valiosa.
columnas_a_eliminar = pct_nulos[pct_nulos > 85].index.tolist()

# Eliminar zip_code: solo muestra los primeros 3 digitos del codigo postal,
# insuficiente para identificar una ubicacion geografica especifica
columnas_a_eliminar.append("zip_code")

columnas_seleccionadas = [col for col in columnas_seleccionadas if col not in columnas_a_eliminar]

print("\nColumnas eliminadas:")
print(columnas_a_eliminar)
print(f"\nColumnas restantes: {len(columnas_seleccionadas)}")

Columnas con mas del 30% de nulos:
dti_joint                      99.942640
annual_inc_joint               99.942415
verification_status_joint      99.942415
il_util                        97.902024
mths_since_rcnt_il             97.654892
inq_fi                         97.591559
open_rv_24m                    97.591559
open_acc_6m                    97.591559
open_rv_12m                    97.591559
inq_last_12m                   97.591559
all_util                       97.591559
max_bal_bc                     97.591559
open_il_6m                     97.591559
open_il_24m                    97.591559
open_il_12m                    97.591559
total_bal_il                   97.591559
total_cu_tl                    97.591559
mths_since_last_record         84.555303
mths_since_last_major_derog    75.015974
mths_since_last_delinq         51.197065
dtype: float64

Columnas eliminadas:
['dti_joint', 'annual_inc_joint', 'verification_status_joint', 'il_util', 'mths_since_rcnt_il', 'inq_fi', 'o

In [6]:
# Identificar columnas de tipo string para definir como tratarlas
string_columns = data_loan[columnas_seleccionadas].select_dtypes(include=["object", "string"]).columns

for col in string_columns:
    print(f"Valores unicos en {col}:")
    print(data_loan[col].unique())
    print()

Valores unicos en term:
[' 36 months' ' 60 months']

Valores unicos en emp_title:
[nan 'Ryder' 'AIR RESOURCES BOARD' ... 'machining Cell Lead'
 'KYC Business Analyst' 'Manager Hotel Operations Oasis ']

Valores unicos en emp_length:
['10+ years' '< 1 year' '1 year' '3 years' '8 years' '9 years' '4 years'
 '5 years' '6 years' '2 years' '7 years' nan]

Valores unicos en home_ownership:
['RENT' 'OWN' 'MORTGAGE' 'OTHER' 'NONE' 'ANY']

Valores unicos en verification_status:
['Verified' 'Source Verified' 'Not Verified']

Valores unicos en purpose:
['credit_card' 'car' 'small_business' 'other' 'wedding'
 'debt_consolidation' 'home_improvement' 'major_purchase' 'medical'
 'moving' 'vacation' 'house' 'renewable_energy' 'educational']

Valores unicos en addr_state:
['AZ' 'GA' 'IL' 'CA' 'OR' 'NC' 'TX' 'VA' 'MO' 'CT' 'UT' 'FL' 'NY' 'PA'
 'MN' 'NJ' 'KY' 'OH' 'SC' 'RI' 'LA' 'MA' 'WA' 'WI' 'AL' 'CO' 'KS' 'NV'
 'AK' 'MD' 'WV' 'VT' 'MI' 'DC' 'SD' 'NH' 'AR' 'NM' 'MT' 'HI' 'WY' 'OK'
 'DE' 'MS' 'TN' 'IA' 

In [7]:
# Clasificar la variable emp_title en categorias de ocupacion
# ya que tiene demasiados valores unicos para usar directamente
def clasificar_titulo(titulo):
    if pd.isna(titulo) or str(titulo).strip() == "":
        return "Unknown"

    titulo_original = str(titulo).strip()
    t = titulo_original.lower()

    # Filtrar valores sin informacion util
    if len(t.replace(" ", "")) <= 2:
        return "Unknown"
    if re.fullmatch(r'[^a-zA-Z]*', titulo_original):
        return "Unknown"
    if re.fullmatch(r'[a-zA-Z]{1,3}', titulo_original.strip()):
        return "Unknown"

    # Normalizar abreviaciones comunes
    abrevs = {
        r'\bsr\.?\b':    'senior',
        r'\bjr\.?\b':    'junior',
        r'\bmgr\.?\b':   'manager',
        r'\basst\.?\b':  'assistant',
        r'\bsvp\.?\b':   'vice president',
        r'\bevp\.?\b':   'executive vice president',
        r'\beng\.?\b':   'engineer',
        r'\btech\.?\b':  'technician',
        r'\bacctg\.?\b': 'accounting',
        r'\badmin\.?\b': 'administrator',
        r'\bspec\.?\b':  'specialist',
        r'\brep\.?\b':   'representative',
        r'\bcoord\.?\b': 'coordinator',
        r'\bdev\.?\b':   'developer',
        r'\bexec\.?\b':  'executive',
        r'\bsupv\.?\b':  'supervisor',
        r'\bdir\.?\b':   'director',
        r'\bpres\.?\b':  'president',
        r'\bgen\.?\b':   'general',
        r'\bmsr\.?\b':   'master',
        r'\bhr\.?\b':    'human resources',
        r'\bsgt\.?\b':   'sergeant',
        r'\blt\.?\b':    'lieutenant',
        r'\bcpt\.?\b':   'captain',
        r'\bdet\.?\b':   'detective',
        r'\binsp\.?\b':  'inspector',
        r'\bap\b':       'accounts payable',
        r'\bar\b':       'accounts receivable',
        r'\bvp\b':       'vice president',
    }
    for pattern, replacement in abrevs.items():
        t = re.sub(pattern, replacement, t)

    # Self employed
    if t.strip() in ["self", "self employed", "self-employed"]:
        return "Self_Employed"

    # Titulos genericos de una sola palabra
    single_word_map = {
        "management":     "Management",
        "administration": "Administrative",
        "administrative": "Administrative",
        "marketing":      "Sales_Retail",
        "sales":          "Sales_Retail",
        "salesman":       "Sales_Retail",
        "saleswoman":     "Sales_Retail",
        "accounting":     "Finance_Legal",
        "finance":        "Finance_Legal",
        "security":       "Government_Military",
        "transportation": "Transportation",
        "engineering":    "Tech",
        "operations":     "Operations",
        "recruiter":      "Administrative",
        "purchasing":     "Operations",
        "consulting":     "Tech",
        "specialist":     "Tech",
        "associate":      "Tech",
        "counselor":      "Social_Community",
        "agent":          "Sales_Retail",
        "inspector":      "Operations",
        "assembler":      "Operations",
        "painter":        "Construction",
        "installer":      "Construction",
        "producer":       "Tech",
        "chemist":        "Tech",
        "planner":        "Tech",
        "editor":         "Tech",
        "labor":          "Operations",
        "laborer":        "Operations",
        "broker":         "Finance_Legal",
    }
    if t.strip() in single_word_map:
        return single_word_map[t.strip()]

    # Keywords de cargos para detectar nombres de empresa
    job_keywords = r"""
        manager|analyst|engineer|specialist|director|officer|
        coordinator|supervisor|assistant|associate|consultant|
        technician|administrator|representative|developer|
        nurse|doctor|teacher|driver|accountant|agent|advisor|
        lead\b|senior|junior|intern|clerk|support|planner|
        researcher|scientist|architect|designer|trainer|
        worker|operator|inspector|auditor|buyer|recruiter|
        teller|estimator|pastor|sergeant|captain|deputy|
        adjuster|assembler|installer|producer|chemist|editor
    """

    # Detectar nombres de empresa por sufijos legales
    if re.search(r'\b(llc|inc\.?|corp\.?|ltd\.?|co\.|plc|l\.p\.)\b', t):
        return "Company_Name"

    # Una sola palabra en mayusculas de 4+ letras es probablemente empresa
    if re.fullmatch(r'[A-Z]{4,}', titulo_original.strip()):
        return "Company_Name"

    # Contiene & pero no es R&D
    if re.search(r'&', titulo_original):
        if not re.search(r'r\s*&\s*d|research\s*&\s*dev|loans\s*&', t):
            return "Company_Name"

    # Empresas conocidas sin keyword de cargo
    known_companies = r"""
        walgreens|starbucks|walmart|amazon|google|apple|microsoft|
        fedex|target|costco|kroger|cvs|
        six\s+flags|kindercare|
        stericycle|maersk|mitre|glenmede|activision|
        verizon|comcast|
        toyota|bmw|honda|
        kaiser(\s+permanente)?|
        bank\s+of\s+america|wells\s+fargo|
        jp\s*morgan(\s+chase)?|
        goldman\s+sachs|morgan\s+stanley|
        lockheed(\s+martin)?|raytheon|boeing|
        northrop|general\s+dynamics|
        ups\b|usps\b|fedex|
        home\s+depot|lowes|
        att\b|sprint\b|t.mobile|
        cigna|aetna|humana|anthem|
        powertrain|seeds\b|luxur|
        department\s+of\s+defense|dept\s+of\s+defense
    """
    if re.search(known_companies, t):
        if not re.search(job_keywords, t, re.VERBOSE):
            return "Company_Name"

    if re.search(r"""
        \bceo\b|\bcfo\b|\bcto\b|\bcoo\b|\bcmo\b|\bchro\b|\bciso\b|
        chief\s+|
        \bpresident\b|\bowner\b|\bfounder\b|\bpartner\b|\bdirector\b|
        vice\s+president|executive\s+director|managing\s+director|
        head\s+of|regional\s+director|national\s+director|
        general\s+manager|\bprincipal\b
    """, t, re.VERBOSE):
        return "Executive"

    if re.search(r"""
        \bmanager\b|\bsupervisor\b|\bsuperintendent\b|\bforeman\b|
        \bcoordinator\b|\blead\b|
        team\s+lead|shift\s+lead|project\s+lead|department\s+head|
        area\s+manager|district\s+manager|branch\s+manager|store\s+manager|
        program\s+manager|project\s+manager|assistant\s+manager|
        senior\s+manager|operations\s+manager
    """, t, re.VERBOSE):
        return "Management"

    if re.search(r"""
        administrative|administrator|
        admin\s+assistant|executive\s+assistant|office\s+assistant|
        office\s+manager|office\s+admin|
        receptionist|secretary|clerical|data\s+entry|records\s+|
        human\s+resources|
        hr\s+(?:generalist|specialist|coordinator|director|manager)|
        talent\s+acquisition|
        payroll\s+(?:specialist|coordinator|administrator)|
        contract\s+(?:specialist|administrator|coordinator)|
        program\s+(?:specialist|administrator)|
        \bplanner\b(?=.*program|.*event|.*urban|.*city|.*project)
    """, t, re.VERBOSE):
        return "Administrative"

    if re.search(r"""
        \bnurse\b|nursing|\brn\b|\blpn\b|\bcna\b|\bcrna\b|
        \bdoctor\b|physician|\bmd\b|surgeon|
        medical|medicine|clinical|clinician|
        dental|dentist|hygienist|orthodon|
        therapist|therapy|\brehab\b|pharmacist|pharmacy|
        radiolog|patholog|oncolog|cardiol|paramedic|\bemt\b|
        healthcare|home\s+health|home\s+care|psycholog|psychiatr|
        caregiver|physical\s+therapy|occupational\s+therapy|
        lab\s+technician|phlebotom|chiropractic|chiropractor|
        \bspeech\b|audiolog|
        health\s+(?:coach|admin|specialist|aide|worker)|
        \boptometr\b|\bophtalm\b|veterinari|vet\s+tech|
        \bcounselor\b(?=.*mental|.*health|.*drug|.*substance|.*rehab|.*addiction)|
        claims\s+(?:nurse|specialist\s+medical)
    """, t, re.VERBOSE):
        return "Healthcare"

    if re.search(r"""
        \bpolice\b|law\s+enforcement|\bdetective\b|
        military|\barmy\b|\bnavy\b|\bmarine\b|air\s+force|coast\s+guard|
        firefighter|fire\s+(?:dept|department|fighter)|
        \bcorrectional\b|corrections\s+officer|\bprobation\b|\bparole\b|
        state\s+of\s+|city\s+of\s+|county\s+of\s+|supreme\s+court|
        federal\s+(?:agent|officer|employee)|
        \bsheriff\b|deputy\s+sheriff|\bdeputy\b|
        \bsergeant\b|\bcaptain\b|\blieutenant\b|
        security\s+officer|security\s+guard|\bguard\b|
        \binvestigator\b|special\s+agent|
        \bofficer\b(?!\s+loan)
    """, t, re.VERBOSE):
        return "Government_Military"

    if re.search(r"""
        accountant|accounting|auditor|\baudit\b|\bcpa\b|bookkeeper|
        \bfinance\b|financial|\bbanker\b|banking|controller|comptroller|
        \btax\b|investment|portfolio|
        loan\s+(?:officer|analyst|review|processor)|
        \bmortgage\b|\bbudget\b|treasury|credit\s+analyst|underwriter|
        financial\s+advisor|financial\s+planner|\bactuar\b|
        \battorney\b|\blawyer\b|\blegal\b|\bparalegal\b|law\s+clerk|
        \bcounsel\b|\bcompliance\b|wealth\s+manag|\bteller\b|
        accounts\s+(?:payable|receivable)|\bbroker\b|
        claims\s+(?:adjuster|analyst|representative|specialist|examiner)|
        insurance\s+(?:adjuster|underwriter|analyst)|
        loan\s+review|\bescrow\b|\btitle\s+officer\b
    """, t, re.VERBOSE):
        return "Finance_Legal"

    if re.search(r"""
        \bteacher\b|teaching|\bprofessor\b|\bfaculty\b|\blecturer\b|
        \binstructor\b|\btrainer\b|\bprincipal\b|\bdean\b|\bprovost\b|
        \beducator\b|\btutor\b|\bschool\b|university|college|\bacademi\b|
        curriculum|\blibrarian\b|special\s+ed|training\s+specialist|
        learning\s+(?:specialist|coordinator|director)
    """, t, re.VERBOSE):
        return "Education"

    if re.search(r"""
        \bsales\b|\bretail\b|\bcashier\b|
        \bsalesman\b|\bsaleswoman\b|\bsalesperson\b|
        account\s+executive|account\s+rep|\bmerchandis\b|\bbuyer\b|
        customer\s+service|client\s+service|telemarket|
        realtor|real\s+estate\s+(?:agent|broker)|
        insurance\s+(?:agent|sales|broker)|business\s+development|
        parts\s+(?:manager|specialist|advisor)|
        marketing\s+(?:specialist|manager|director|coordinator|analyst|planner|associate)|
        media\s+(?:planner|buyer|specialist)|brand\s+(?:manager|specialist)|
        \brecruiter\b|staffing\s+|\bclerk\b(?!\s+law)|service\s+advisor|
        \bagent\b(?=\s+insurance|\s+real|\s+travel|\s+sales|$)
    """, t, re.VERBOSE):
        return "Sales_Retail"

    if re.search(r"""
        software|developer|programmer|\bit\b|information\s+tech|
        \bdatabase\b|data\s+|\bcyber\b|infosec|network\s+|systems\s+admin|
        devops|cloud\s+|web\s+|front.end|back.end|full.stack|
        machine\s+learning|artificial\s+intel|\bai\b|\bml\b|
        desktop\s+support|help\s+desk|\bengineer\b|
        systems\s+engineer|project\s+engineer|electrical\s+engineer|
        mechanical\s+engineer|chemical\s+engineer|process\s+engineer|
        research\s+(?:engineer|developer|scientist|associate|analyst|specialist)|
        batch\s+processing|
        (?:business|systems|data|senior|global|pricing|governance)\s+analyst|
        \banalyst\b|graphic\s+designer|\bux\b|\bui\b|
        product\s+(?:manager|owner|designer)|\barchitect\b|solution\s+architect|
        \bscientist\b|\bresearcher\b|\bchemist\b|biochem|biolog|
        \bgeologist\b|geograph|\bproducer\b|\beditor\b|content\s+|\bplanner\b|
        senior\s+associate|\bconsultant\b|consulting\s+|
        specialist(?=\s+contract|\s+program|\s+technical|\s+it\b|\s+data)
    """, t, re.VERBOSE):
        return "Tech"

    if re.search(r"""
        \boperator\b|\bmechanic\b|\btechnician\b|\bmaintenance\b|
        \bwarehouse\b|supply\s+chain|\binventory\b|
        manufactur|production|\bassembl\b|\bmachinist\b|quality\s+control|
        \bwelder\b|\bfabricat\b|\blaborer?\b|material\s+handler|
        \bshipping\b|\breceiving\b|custodian|janitorial|housekeep|
        landscap|groundskeep|\brigger\b|\bmillwright\b|
        field\s+(?:service|technician|engineer)|electronic\s+technician|
        repair\s+technician|service\s+technician|\bmining\b|\bdrilling\b|\bmine\b|
        line\s+(?:worker|operator|technician)|general\s+labor|skilled\s+labor|
        support\s+(?:services|specialist)|engineering\s+aide|
        \bpipefitter\b|\bironworker\b|\binspector\b|\bassembler\b
    """, t, re.VERBOSE):
        return "Operations"

    if re.search(r"""
        construct|\bcarpenter\b|\belectrician\b|\bplumber\b|\bhvac\b|
        \broofer\b|roofing|\bbuilder\b|civil\s+engineer|structural\s+engineer|
        \bsurveyor\b|\bconcrete\b|\bpipefitter\b|\bmason\b|
        \bpainter\b(?=.*house|.*build|.*construct|$)|
        \binstaller\b|installation|\bdrywaller\b|\bframer\b|
        \bestimator\b(?=.*construct|.*build)
    """, t, re.VERBOSE):
        return "Construction"

    if re.search(r"""
        \bdriver\b|\btruck\b|\bcdl\b|\bdelivery\b|\bcourier\b|
        \bpilot\b|aviation|airline|\bflight\b|
        \bdispatcher\b|\btransit\b|bus\s+driver|\bpostal\b|
        letter\s+carrier|mail\s+carrier|\bcarrier\b|
        \bconductor\b|\btransportation\b
    """, t, re.VERBOSE):
        return "Transportation"

    if re.search(r"""
        \bchef\b|\bcook\b|culinary|\bkitchen\b|
        \bserver\b|\bwaiter\b|\bwaitress\b|
        bartender|barista|\bhotel\b|hospitality|
        restaurant|food\s+service|catering|\bconcierge\b|\blodge\b|\bslot\b
    """, t, re.VERBOSE):
        return "Hospitality"

    if re.search(r"""
        social\s+work|social\s+worker|case\s+(?:manager|worker)|
        nonprofit|non.profit|community\s+|outreach|\bwelfare\b|
        substance\s+abuse|\bpastor\b|\bchaplain\b|\bminister\b|
        \bpriest\b|\bclergy\b|\bdeacon\b|\bmissionary\b|\bbishop\b|
        \bcounselor\b
    """, t, re.VERBOSE):
        return "Social_Community"

    if re.search(r"""
        self.employ|freelance|independent\s+contractor|
        own\s+business|business\s+owner|entrepreneur
    """, t, re.VERBOSE):
        return "Self_Employed"

    if re.search(r"""
        \bspecialist\b|\bassociate\b|senior\s+associate|\bconsultant\b
    """, t, re.VERBOSE):
        return "Tech"

    return "Other"

In [ ]:
# Aplicar la funcion de clasificacion a la columna emp_title
data_loan["emp_title_cat"] = data_loan["emp_title"].apply(clasificar_titulo)

print("Distribucion de categorias de empleo:")
print(data_loan["emp_title_cat"].value_counts())

print(f"\nRegistros en categoria Other: {(data_loan['emp_title_cat'] == 'Other').sum()} "
      f"({(data_loan['emp_title_cat'] == 'Other').mean() * 100:.1f}%)")

print("\nTop 30 titulos mas frecuentes en categoria Other:")
print(
    data_loan[data_loan["emp_title_cat"] == "Other"]["emp_title"]
    .dropna()
    .value_counts()
    .head(30)
    .to_string()
)

In [ ]:
# term: extraer el numero de meses
data_loan["term"] = (
    data_loan["term"]
    .str.extract(r"(\d+)")
    .squeeze()
    .astype(int)
)

# emp_length: limpiar y convertir a numero
data_loan["emp_length"] = (
    data_loan["emp_length"]
    .str.replace(r"\s*years?", "", regex=True)
    .str.replace("< 1", "0")
    .str.replace("10+", "10")
    .str.strip()
    .pipe(pd.to_numeric, errors="coerce")
)

# int_rate y revol_util: quitar el simbolo de porcentaje
for col in ["int_rate", "revol_util"]:
    if data_loan[col].dtype == object:
        data_loan[col] = pd.to_numeric(
            data_loan[col].str.replace("%", "", regex=False).str.strip(),
            errors="coerce"
        )

print("Dtypes despues de conversion:")
print(data_loan[["term", "emp_length", "int_rate", "revol_util"]].dtypes)

In [ ]:
# Variables categoricas nominales: se usa OHE para evitar
# que el modelo interprete un orden entre las categorias
categorical_ohe = [
    "home_ownership",
    "verification_status",
    "purpose",
    "initial_list_status",
    "application_type",
    "emp_title_cat",
]

# addr_state se agrupa por region para reducir de 51 a 4 categorias
region_map = {
    "Northeast": ["CT", "ME", "MA", "NH", "RI", "VT", "NY", "NJ", "PA"],
    "South":     ["DE", "MD", "DC", "VA", "WV", "NC", "SC", "GA", "FL",
                  "KY", "TN", "AL", "MS", "AR", "LA", "OK", "TX"],
    "Midwest":   ["OH", "IN", "IL", "MI", "WI", "MN", "IA", "MO", "ND",
                  "SD", "NE", "KS"],
    "West":      ["MT", "ID", "WY", "CO", "NM", "AZ", "UT", "NV", "WA",
                  "OR", "CA", "AK", "HI"],
}
state_to_region = {state: region for region, states in region_map.items() for state in states}
data_loan["addr_region"] = data_loan["addr_state"].map(state_to_region).fillna("Other")

categorical_ohe.append("addr_region")

# Aplicar OHE
data_loan = pd.get_dummies(data_loan, columns=categorical_ohe, drop_first=False)

print("Columnas OHE generadas:")
ohe_prefixes = ["home_ownership_", "verification_status_", "purpose_",
                "initial_list_status_", "application_type_", "emp_title_cat_", "addr_region_"]
ohe_cols = [c for c in data_loan.columns if any(c.startswith(p) for p in ohe_prefixes)]
print(ohe_cols)